# Step 0.1 — Preprocessing: land mask and MIRCA2000 crop masks on the EC-Earth3 grid (global, no threshold)

The SVM grid is the **EC-Earth3 grid** (0.7°, global): the grid of ERA5 regridded with CDO (training climate), of
EC-Earth3 historical and of the four AMOC runs (delta method). It is read by `crop_utils.svm_grid()` (cell centres and
edges from an EC-Earth3 file); every notebook uses it, there is no hand-made box.

1. **Land mask**: Natural Earth `land_10` at 5 arc-min (pixel-centre test); an EC-Earth3 cell is land if ≥ 25 % of
   its 5-arc-min pixels are land (same rule as the former 1° grid). The same mask is applied to all the climate data
   (ERA5, EC-Earth3 historical, runs: all on this grid) and to the crop masks.
2. **MIRCA2000, no threshold**: an EC-Earth3 cell is *cropped* for a crop if MIRCA2000 records **any** harvested area
   (> 0) in one of its 5-arc-min pixels (`CELL_SPECIFIC_CROPPING_CALENDARS`, column `area`, summed over the sub-crops).
   **Calendar** (sowing, maturity; DOY) = circular mean of the sub-crops' start and end months, weighted by their
   harvested area, over **all** the sub-crop records of the cell (no threshold either).
3. **Wheat**: each sub-crop record is *winter* wheat if sown in autumn–early winter, *spring* wheat otherwise, the
   season being taken **relative to the hemisphere** (southern hemisphere: months shifted by 6), as the sowing
   windows of Waha et al. (2012) / Minoli et al. (2022, GGCMI-LPJmL: winter wheat sown after 15 September in the
   northern hemisphere, after 31 March in the southern one):

   | | winter wheat | spring wheat |
   |---|---|---|
   | northern hemisphere | sown Sep–Jan | sown Feb–Aug |
   | southern hemisphere | sown Mar–Jul | sown Aug–Feb |

   so that every wheat record is classified (with the former Europe rule, Sep–Dec / Feb–May, 86 % of the southern
   hemisphere's wheat and 14 Mha of northern spring wheat sown in June–July were dropped). A third crop **`Wheat`**
   keeps winter and spring wheat together: presence = any wheat in the cell; calendar = the one of the season with the
   larger harvested area in the cell (a mean of an autumn and a spring sowing would be meaningless).

ERA5 needs no step here: it is already on the EC-Earth3 grid (`ERA5_daily_on_ecearth3grid`); the BIO notebooks read it
through the configuration and apply the land mask.

Output folder: `1_SVM/0_preprocessing_masks/outputs_ecearth3/` (`crop_utils.SVM_GRID_DIR`):
`land_mask_ecearth3.nc`, `mirca2000_no_threshold_ecearth3_<Crop>_<rf|ir>.nc` (presence, harvested area, sowing and
maturity DOY).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, regionmask
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
CAL_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "growing_periods_listed" / "CELL_SPECIFIC_CROPPING_CALENDARS.TXT.gz"
OUT = cu.SVM_GRID_DIR
OUT.mkdir(parents=True, exist_ok=True)

LAND_THR = 0.25                                   # land cell: >= 25 % of its 5-arc-min pixels are land
GRID = cu.svm_grid(CFG)
NLAT, NLON = GRID["lat"].size, GRID["lon"].size
NCELL = NLAT * NLON
print(f"SVM grid (EC-Earth3): {NLAT} x {NLON} = {NCELL} cells | lat {GRID['lat'][0]:.2f}..{GRID['lat'][-1]:.2f}, "
      f"lon {GRID['lon'][0]:.2f}..{GRID['lon'][-1]:.2f}")

## 1. Land mask: Natural Earth at 5 arc-min, EC-Earth3 cell land if ≥ 25 % land

In [ ]:
NROW, NCOL = 2160, 4320                           # MIRCA2000 5-arc-min grid (row 1 = north, col 1 = 180 W)
lat5 = 90 - (np.arange(NROW) + 0.5) / 12
lon5 = -180 + (np.arange(NCOL) + 0.5) / 12
I5, J5 = cu.cell_of(lat5[:, None] * np.ones((1, NCOL)), np.ones((NROW, 1)) * lon5[None, :], GRID)
CELL5 = (I5 * NLON + J5).astype(np.int64)         # (2160, 4320): EC-Earth3 cell of each 5-arc-min pixel

land5 = regionmask.defined_regions.natural_earth_v5_0_0.land_10.mask(lon5, lat5).notnull().values
n_pix = np.bincount(CELL5.ravel(), minlength=NCELL)
n_land = np.bincount(CELL5.ravel(), weights=land5.ravel().astype(float), minlength=NCELL)
land_frac = (n_land / np.maximum(n_pix, 1)).reshape(NLAT, NLON)
LAND = land_frac >= LAND_THR

xr.Dataset({"land_mask": (("lat", "lon"), LAND.astype("int8"), {"long_name": f"land (>= {LAND_THR:.0%} of the 5-arc-min pixels)"}),
            "land_frac": (("lat", "lon"), land_frac.astype("float32"), {"long_name": "share of the 5-arc-min pixels on land (Natural Earth 1:10m)"})},
           coords={"lat": GRID["lat"], "lon": GRID["lon"]},
           attrs={"grid": "EC-Earth3 gr (from crop_utils.svm_grid)"}).to_netcdf(OUT / "land_mask_ecearth3.nc")
print(f"5-arc-min pixels per EC-Earth3 cell: {n_pix.min()}-{n_pix.max()} (median {int(np.median(n_pix))})")
print(f"land cells: {int(LAND.sum())} of {NCELL} ({100 * LAND.mean():.1f} %)")

## 2. MIRCA2000 per EC-Earth3 cell: presence (any harvested area) and calendar (all sub-crop records)

In [ ]:
CAL_CLASSES = {"Wheat": (1, 27), "Maize": (2, 28), "Rice": (3, 29), "Soybean": (8, 34), "Potato": (10, 36)}
# (irrigated code, rainfed code); rainfed = irrigated + 26

cols = ["cell", "row", "col", "lat", "lon", "crop", "subcrop", "area", "start", "end"]
cal = pd.read_csv(CAL_FILE, sep="\t", compression="gzip", usecols=range(10), header=0, names=cols)
cal = cal[cal.crop.isin([c for v in CAL_CLASSES.values() for c in v])].copy()
cal["k"] = CELL5[cal.row.values - 1, cal.col.values - 1]          # EC-Earth3 cell of the record's 5-arc-min pixel
cal["irr"] = np.where(cal.crop <= 26, "ir", "rf")
cal["crop_name"] = cal.crop.map({c: k for k, v in CAL_CLASSES.items() for c in v})
print("sub-crop records:", len(cal))


def circ(m):
    return 2 * np.pi * (np.asarray(m, float) - 1) / 12.0      # month -> angle


def month_to_doy(m):
    return (np.asarray(m, float) - 1) * 365.0 / 12.0 + 15.2   # mid-month DOY


def cell_month(k, area, month):
    # area-weighted circular mean month (1..13) of the records of each cell
    sx = np.bincount(k, weights=area * np.cos(circ(month)), minlength=NCELL)
    sy = np.bincount(k, weights=area * np.sin(circ(month)), minlength=NCELL)
    return np.mod(np.arctan2(sy, sx), 2 * np.pi) * 12 / (2 * np.pi) + 1.0


# wheat season of each record, relative to the hemisphere: southern hemisphere months shifted by 6
m_hemi = np.where(cal.lat.values >= 0, cal.start.values, (cal.start.values + 5) % 12 + 1)
cal["wheat_season"] = np.where(np.isin(m_hemi, [9, 10, 11, 12, 1]), "winter", "spring")


def cell_products(gs):
    # harvested area, sowing and maturity DOY per cell (2D) from the sub-crop records gs
    k, a = gs.k.values, gs.area.values
    area = np.bincount(k, weights=a, minlength=NCELL).reshape(NLAT, NLON)
    start_m, end_m = cell_month(k, a, gs.start.values), cell_month(k, a, gs.end.values)
    sow = month_to_doy(start_m).reshape(NLAT, NLON)
    mat = sow + (np.mod(end_m - start_m, 12.0) * 365.0 / 12.0).reshape(NLAT, NLON)   # may exceed 365
    return area, sow, mat


rows = []
for crop, g_all in cal.groupby("crop_name"):
    for irr, g in g_all.groupby("irr"):
        if crop == "Wheat":
            P = {s: cell_products(g[g.wheat_season == s]) for s in ("winter", "spring")}
            aw, asp = P["winter"][0], P["spring"][0]
            dominant = np.where(aw >= asp, 0, 1)[None]                         # calendar of the larger season
            comb = (aw + asp,) + tuple(np.choose(dominant, [P["winter"][i][None], P["spring"][i][None]])[0] for i in (1, 2))
            products = {"Wheat-winter": P["winter"], "Wheat-spring": P["spring"], "Wheat": comb}
            n_rec = {"Wheat-winter": g.wheat_season.eq("winter").sum(), "Wheat-spring": g.wheat_season.eq("spring").sum(), "Wheat": len(g)}
            src = {"Wheat-winter": g[g.wheat_season == "winter"].area.sum(), "Wheat-spring": g[g.wheat_season == "spring"].area.sum(), "Wheat": g.area.sum()}
        else:
            products, src = {crop: cell_products(g)}, {crop: g.area.sum()}
        for name, (area2, sow, mat) in products.items():
            tag = f"{name}_{irr}"
            pres = (area2 > 0) & LAND                                          # no threshold, land cells only
            xr.Dataset({
                "presence": (("lat", "lon"), np.where(LAND, pres.astype(float), np.nan),
                             {"long_name": f"MIRCA2000 presence, no threshold (harvested area > 0), {tag}"}),
                "area_ha": (("lat", "lon"), np.where(LAND, area2, np.nan).astype("float32"),
                            {"long_name": "MIRCA2000 harvested area in the cell", "units": "ha"}),
                "sowing_doy": (("lat", "lon"), np.where(pres, sow, np.nan).astype("float32"),
                               {"long_name": "sowing DOY, area-weighted circular mean over all sub-crop records"}),
                "maturity_doy": (("lat", "lon"), np.where(pres, mat, np.nan).astype("float32"),
                                 {"long_name": "maturity DOY (> 365: harvest the next year)"}),
            }, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"description": f"MIRCA2000 (no threshold) on the EC-Earth3 grid -- {tag}"}
            ).to_netcdf(OUT / f"mirca2000_no_threshold_ecearth3_{tag}.nc")
            rows.append({"crop": tag, "cropped cells": int(pres.sum()), "% of land cells": round(100 * pres.sum() / LAND.sum(), 1),
                         "MIRCA area (Mha)": round(src[name] / 1e6, 2),
                         "area kept on land cells (%)": round(100 * area2[LAND].sum() / max(src[name], 1), 2)})
summary = pd.DataFrame(rows).set_index("crop")
summary.to_csv(OUT / "mirca2000_no_threshold_ecearth3_summary.csv")
with pd.option_context("display.width", 200, "display.max_columns", 10):
    display(summary)

## 3. Checks

`area kept on land cells` above: share of MIRCA's harvested area that falls in land cells (the rest is in cells with
< 25 % land: small islands, some coasts). Below: land mask, presence and calendar of one crop, and the calendar at a
few places.

In [ ]:
CHECK = "Wheat-winter_rf"
d = xr.open_dataset(OUT / f"mirca2000_no_threshold_ecearth3_{CHECK}.nc")
fig, axes = plt.subplots(2, 2, figsize=(15, 7))
for ax, (v, t, cm) in zip(axes.ravel(), [(xr.DataArray(LAND, coords=d.presence.coords), "land mask", "Greens"),
                                         (d.presence, f"presence {CHECK}", "YlGn"),
                                         (d.sowing_doy, f"sowing DOY {CHECK}", "plasma"),
                                         (d.maturity_doy, f"maturity DOY {CHECK}", "viridis")]):
    v.plot(ax=ax, cmap=cm); ax.set_title(t)
plt.tight_layout(); plt.savefig(OUT / f"check_{CHECK}.png", dpi=90); plt.show()

import datetime
def doy2str(x):
    return "NaN" if not np.isfinite(x) else (datetime.date(2001, 1, 1) + datetime.timedelta(days=int(x) % 365)).strftime("%b %d")
places = {"France (47N, 2E)": (47, 2), "Ukraine (49N, 32E)": (49, 32), "Kansas (38N, 98W)": (38, -98),
          "Punjab (30N, 75E)": (30, 75), "North China (36N, 115E)": (36, 115), "Argentina (34S, 62W)": (-34, -62),
          "New South Wales (33S, 147E)": (-33, 147), "Saskatchewan (52N, 106W)": (52, -106), "Kazakhstan (52N, 70E)": (52, 70)}
for crop in ("Wheat-winter_rf", "Wheat-spring_rf", "Wheat_rf", "Maize_rf"):
    d = xr.open_dataset(OUT / f"mirca2000_no_threshold_ecearth3_{crop}.nc")
    print(crop)
    for name, (la, lo) in places.items():
        p = d.sel(lat=la, lon=lo, method="nearest")
        print(f"  {name:24s} presence {float(p.presence):.0f}  sowing {doy2str(float(p.sowing_doy)):7s}  "
              f"maturity {doy2str(float(p.maturity_doy))}  ({float(p.area_ha):.0f} ha)")